# SNP & Variant Identification in *E. coli* — End-to-End NGS Pipeline

**Author:** Szonja Fekete  
**Dataset:** *E. coli* clinical isolate EC_UTI_2023_001 (urinary tract infection, NL, 2023)  
**Reference:** *E. coli* K-12 MG1655 — NC_000913.3 (4,641,652 bp)  
**Platform:** Illumina NovaSeq 6000 · 150bp paired-end · Galaxy workflow  

---

## Background & Biological Motivation

Antibiotic resistance in gram-negative bacteria — particularly *Escherichia coli* — is a major clinical challenge. Resistance mechanisms often arise through **point mutations** (SNPs) in drug-target genes or in regulatory genes that control the expression of efflux pumps.

In this project, we take a clinical *E. coli* isolate from a urinary tract infection and use **Next-Generation Sequencing (NGS)** to:
1. Assess raw read quality and trim adapters (FastQC, Trimmomatic)
2. Align trimmed reads to the reference genome (BWA-MEM)
3. Compute coverage depth and alignment statistics (SAMtools)
4. Call variants using a Bayesian haplotype model (FreeBayes)
5. Annotate variants with predicted functional effects (SnpEff)
6. Identify clinically relevant resistance mutations (curated CARD/ResFinder DB)

### Key Resistance Genes to Watch
| Gene | Product | Resistance Mechanism |
|------|---------|----------------------|
| `gyrA` | DNA gyrase subunit A | Fluoroquinolone — altered drug target (QRDR) |
| `rpoB` | RNA polymerase β | Rifampicin — altered drug target |
| `marR` | Multiple antibiotic resistance regulator | MDR — derepresses AcrAB-TolC efflux |
| `acrA` | Efflux pump MFP subunit | MDR — enhanced drug efflux |
| `tolC` | Outer-membrane efflux channel | MDR — altered channel specificity |
| `ompC` | Outer-membrane porin C | Broad — reduced permeability |


## 0 — Environment Setup

In [ ]:
import sys, os, json
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import seaborn as sns
from IPython.display import Image, display

matplotlib.rcParams['figure.dpi'] = 120

# Paths (works when notebook is run from /notebooks/ or project root)
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
BASE_DIR     = os.path.dirname(NOTEBOOK_DIR)
DATA_DIR     = os.path.join(BASE_DIR, 'data')
RESULTS_DIR  = os.path.join(BASE_DIR, 'results')
FIG_DIR      = os.path.join(BASE_DIR, 'figures')

print('Python', sys.version.split()[0])
print('pandas', pd.__version__, '| numpy', np.__version__, '| matplotlib', matplotlib.__version__)
print('Base directory:', BASE_DIR)

## 1 — Galaxy Workflow: FastQC → Trimmomatic → BWA-MEM → FreeBayes → SnpEff

The complete bioinformatics workflow was executed on the **usegalaxy.eu** server.
The workflow file `galaxy_workflow/ecoli_variant_calling.ga` can be imported directly into any Galaxy instance.

### Step 1A: Raw Read Quality Control (FastQC v0.12.1)

FastQC was run on both R1 and R2 FASTQ files before trimming.

**Key metrics from FastQC:**
- Total sequences: 1,571,044 per end (3,142,088 total reads)
- Sequence length: 150 bp
- %GC: 50.7% ✓ (matches *E. coli* K-12 expected value)
- %Q30: **87.4%** — acceptable; 3′ quality drop visible (normal for Illumina)
- Adapter content: Low TruSeq PE adapters detected in <2% of reads

In [ ]:
# Load QC metrics
qc_df = pd.read_csv(os.path.join(DATA_DIR, 'per_base_quality.csv'))
with open(os.path.join(DATA_DIR, 'qc_summary.json')) as f:
    qc_stats = json.load(f)

print('QC Summary')
print('─' * 42)
for key in ['raw_total_reads','raw_pct_q30','raw_mean_quality',
            'trim_total_reads','trim_surviving_pct','trim_pct_q30','trim_mean_quality']:
    print(f"  {key:<30}: {qc_stats[key]}")
print()
qc_df.head(5)

In [ ]:
# Display per-base quality figure (generated by scripts/generate_figures.py)
display(Image(os.path.join(FIG_DIR, 'per_base_quality.png')))

**Interpretation:**
- **Before trimming** (left): classic Illumina quality profile — excellent in the 1–120 bp window, dropping below Q28 after position 130. This 3′ degradation is caused by signal interference in longer clusters and is expected.
- **After Trimmomatic** (right): uniform high-quality reads. The SLIDINGWINDOW:4:15 parameter removed low-quality tails, raising mean quality from 35.8 → 37.2 and %Q30 from 87.4% → **93.1%**.
- Both plots are colour-coded by the IQR percentile band (Q25–Q75 box, Q10–Q90 whiskers).

### Step 1B: GC Content Distribution

In [ ]:
gc_df = pd.read_csv(os.path.join(DATA_DIR, 'gc_content.csv'))

fig, ax = plt.subplots(figsize=(8, 4))
ax.fill_between(gc_df['gc_pct'], gc_df['theoretical'], alpha=0.3, color='steelblue', label='Theoretical (E. coli K-12)')
ax.plot(gc_df['gc_pct'], gc_df['theoretical'], 'b--', linewidth=1.2)
ax.fill_between(gc_df['gc_pct'], gc_df['actual'], alpha=0.4, color='coral', label='Observed (EC_UTI_2023_001)')
ax.plot(gc_df['gc_pct'], gc_df['actual'], 'r-', linewidth=1.8)
ax.axvline(50.7, color='gold', linewidth=1, linestyle=':')
ax.set_xlabel('GC Content (%)')
ax.set_ylabel('% of Reads')
ax.set_title('Per-Sequence GC Content — Observed vs Theoretical')
ax.set_xlim(25, 75)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print('Observed GC peak:', gc_df.loc[gc_df["actual"].idxmax(), "gc_pct"], '%')

**Interpretation:** The observed GC distribution closely tracks the theoretical curve for *E. coli* K-12 MG1655 (50.7% GC). No bimodal peaks indicating contamination or major structural variation. ✓

### Step 2: Adapter Trimming with Trimmomatic v0.39

```bash
# Galaxy-equivalent command
trimmomatic PE \
  EC_UTI_2023_001_R1.fastq.gz EC_UTI_2023_001_R2.fastq.gz \
  R1_paired.fastq.gz R1_unpaired.fastq.gz \
  R2_paired.fastq.gz R2_unpaired.fastq.gz \
  ILLUMINACLIP:TruSeq3-PE.fa:2:30:10 \
  LEADING:3 TRAILING:3 \
  SLIDINGWINDOW:4:15 \
  MINLEN:36
```

| Parameter | Value | Rationale |
|-----------|-------|----------|
| ILLUMINACLIP | TruSeq3-PE, seed=2 | Remove Illumina TruSeq PE adapters |
| LEADING / TRAILING | 3 | Clip low-quality leading/trailing bases |
| SLIDINGWINDOW | 4:15 | Drop window if avg quality < Q15 |
| MINLEN | 36 | Discard reads shorter than 36bp |

**Result:** 97.3% of reads survived trimming (3,056,442/3,142,088).

## 2 — Alignment with BWA-MEM v0.7.17

BWA-MEM is the standard algorithm for aligning 70–1000 bp reads to a large reference genome. It uses a **seed-and-extend** approach based on the Burrows-Wheeler Transform.

```bash
# Reference indexing
bwa index NC_000913.3.fasta

# Paired-end alignment with read group tags (required for variant calling)
bwa mem -M \
  -R '@RG\tID:EC_UTI_2023_001\tSM:EC_UTI_2023_001\tPL:ILLUMINA\tLB:Lib001' \
  NC_000913.3.fasta \
  R1_paired.fastq.gz R2_paired.fastq.gz \
  | samtools sort -o EC_UTI_2023_001.sorted.bam

# Mark duplicates
samtools markdup EC_UTI_2023_001.sorted.bam EC_UTI_2023_001.dedup.bam
samtools index EC_UTI_2023_001.dedup.bam
```

In [ ]:
# Load alignment statistics
with open(os.path.join(DATA_DIR, 'alignment_stats.json')) as f:
    align_stats = json.load(f)

print('Alignment Statistics (SAMtools flagstat)')
print('─' * 50)
for key, val in align_stats.items():
    print(f'  {key:<30}: {val}')

In [ ]:
# Coverage depth across the genome
display(Image(os.path.join(FIG_DIR, 'genome_coverage.png')))

**Interpretation of coverage plot:**
- **Mean depth: 48.7×** — excellent for bacterial variant calling (target: ≥ 20×)
- Coverage is highly uniform across the 4.64 Mb genome (expected for Illumina short-read sequencing)
- A **low-coverage trough** is visible near position 4.17 Mb — this corresponds to the rRNA operons (16S/23S rRNA), where repetitive sequences reduce unique mapping
- Vertical colour lines mark the positions of key resistance genes: **gyrA** (coral), **rpoB** (teal), **marR** (amber), **acrA** (blue), **tolC** (violet)
- 98.94% of the genome is covered at ≥ 20× — adequate for confident variant calling

## 3 — Variant Calling with FreeBayes v1.3.6

FreeBayes uses a **Bayesian statistical model** to detect SNPs and small indels from the pileup of aligned reads. Unlike GATK HaplotypeCaller, FreeBayes treats the haplotype jointly, making it particularly suitable for bacterial genomes.

**Key parameters:**
```
--min-mapping-quality 20     # Ignore reads with mapping quality < 20
--min-base-quality 20        # Ignore bases with Phred quality < 20
--min-alternate-fraction 0.05 # Consider alleles with ≥ 5% frequency
--min-alternate-count 3      # Require ≥ 3 supporting reads per allele
--min-coverage 5             # Skip positions with < 5× depth
--ploidy 1                   # Haploid (bacterial) mode
```

> 💡 **Why ploidy=1?** Bacteria are haploid. Setting ploidy to 1 tells FreeBayes to expect each locus to carry at most one allele, making it more sensitive for fixed mutations at high allele frequency.

In [ ]:
# Load annotated variants
var_df = pd.read_csv(os.path.join(RESULTS_DIR, 'variants_annotated.csv'))
print(f'Total variants: {len(var_df)}')
print()
var_df[['GENE','POS','REF','ALT','QUAL','DP','AF','AA_CHANGE','EFFECT','CLINICAL_NOTE']].head(10)

In [ ]:
# Quality filter: QUAL >= 200, DP >= 10, AF >= 0.90
filt_df = var_df[(var_df['QUAL'] >= 200) & (var_df['DP'] >= 10) & (var_df['AF'] >= 0.90)].copy()
print(f'After quality filter: {len(filt_df)} variants')
print()

# Summary by functional class
print('Variant counts by functional class:')
print(filt_df['FUNCTIONAL_CLASS'].value_counts().to_string())

In [ ]:
# VAF distribution figures
display(Image(os.path.join(FIG_DIR, 'variant_vaf.png')))

**Interpretation:**
- **Left panel:** All variants cluster at VAF > 88%, consistent with a clonal bacterial population. This is expected — in a pure culture of a single *E. coli* strain, every cell carries the same mutation (VAF ≈ 100%). Slightly sub-100% VAFs reflect mapping uncertainty and PCR noise.
- **Right panel:** High-quality variants (key resistance mutations **gyrA**, **rpoB**, **marR**, **ompC**) show both high QUAL scores (> 700) and high VAFs (> 95%), giving high confidence in these calls.
- The `STOP_GAINED` variant in `ompC` has a lower QUAL (541) but remains well above the 200-threshold — still a confident call.

## 4 — Variant Annotation with SnpEff v5.1d

SnpEff annotates each variant with:
- **Functional effect** (missense, synonymous, stop_gained, intergenic, UTR, …)
- **Gene name** and **transcript ID**
- **HGVS notation** (protein-level: p.Asp87Asn, nucleotide-level: c.259G>A)
- **Putative impact** (HIGH, MODERATE, LOW, MODIFIER)

```bash
# Galaxy SnpEff annotation step
snpEff -v Escherichia_coli_k_12 filtered_variants.vcf > annotated.vcf
```

In [ ]:
# Annotation breakdown
display(Image(os.path.join(FIG_DIR, 'variant_annotation.png')))

In [ ]:
# Nonsynonymous variants in detail
ns_df = filt_df[filt_df['FUNCTIONAL_CLASS'].isin(['NONSYNONYMOUS','NONSENSE'])].copy()
ns_df = ns_df[['GENE','POS','REF','ALT','AA_CHANGE','QUAL','AF','CLINICAL_NOTE']]
ns_df['AF%'] = (ns_df['AF'] * 100).round(1)
ns_df.drop('AF', axis=1, inplace=True)
ns_df

## 5 — Antibiotic Resistance Profile

Cross-referencing variants against the **Comprehensive Antibiotic Resistance Database (CARD)** and **ResFinder** literature.

In [ ]:
display(Image(os.path.join(FIG_DIR, 'resistance_summary.png')))

**Clinical Interpretation of Resistance Findings:**

| Gene | Mutation | Drug | Clinical Significance |
|------|----------|------|---------------------|
| `gyrA` | D87N + A90V | Fluoroquinolones (ciprofloxacin) | **Double QRDR mutation** — high-level resistance (MIC ≥ 32× fold increase); ciprofloxacin treatment contraindicated |
| `rpoB` | S531L | Rifampicin | **Hotspot mutation** — very high-level rifampicin resistance (256× MIC increase); RNA polymerase structural change |
| `marR` | G103S | Multiple drugs | **MDR regulator** — loss-of-function in MarR derepresses MarA, which upregulates AcrAB-TolC efflux and reduces porin expression |
| `acrA` + `tolC` | I355V, L457P | Multiple drugs | **Efflux pump mutations** — may enhance substrate range of AcrAB-TolC |
| `ompC` | K163* (stop) | Broad spectrum | **Porin loss** — nonsense mutation truncates OmpC; reduces outer-membrane permeability to β-lactams, fluoroquinolones |

> ⚠️ **Clinical note:** This isolate shows a **multidrug-resistant (MDR) profile** involving fluoroquinolones, rifampicin, and broad-spectrum mechanisms. The double *gyrA* mutation at QRDR positions 87 + 90 is a well-characterised high-level resistance genotype. This strain would likely be resistant to first-line UTI antibiotics; susceptibility testing with other agents (e.g. nitrofurantoin, fosfomycin) would be recommended.

## 6 — VCF Inspection

The FreeBayes → SnpEff output is stored in standard VCF 4.2 format.

In [ ]:
vcf_path = os.path.join(RESULTS_DIR, 'variants.vcf')
with open(vcf_path) as f:
    lines = f.readlines()

# Print header + first 3 data lines
for line in lines[:20]:
    if line.startswith('##') and len(line) > 80:
        print(line[:78] + '…')
    else:
        print(line, end='')

## 7 — Summary & Conclusions

### What this project demonstrates

1. **NGS Quality Control:** Understanding of FastQC metrics and how to interpret Illumina quality score profiles; use of Trimmomatic with appropriate parameters for PE data
2. **Reference Alignment:** BWA-MEM setup including read group tagging (critical for GATK/FreeBayes compatibility), PCR duplicate marking, and coverage QC
3. **Variant Calling:** FreeBayes configuration for haploid bacterial genomes; understanding of key parameters (min-AF, min-DP, ploidy)
4. **Variant Filtering & Annotation:** Hard-filter strategy (QUAL, DP, AF thresholds) and SnpEff functional annotation
5. **Biological Interpretation:** Cross-referencing variants against resistance databases and translating genomic findings into clinical context

### Pipeline Summary

| Step | Tool | Key Output |
|------|------|------------|
| QC (raw) | FastQC v0.12.1 | %Q30 = 87.4%, no major failures |
| Trimming | Trimmomatic v0.39 | 97.3% reads retained, %Q30 → 93.1% |
| Alignment | BWA-MEM v0.7.17 | 98.57% mapping rate, 48.7× mean coverage |
| Coverage QC | SAMtools + BEDtools | 98.94% genome covered ≥ 20× |
| Variant calling | FreeBayes v1.3.6 | 13 raw variants called |
| Filtering | VCFfilter | 11 high-quality variants retained |
| Annotation | SnpEff v5.1d | 8 coding variants; 7 match resistance DB |

### Key Findings

- **7 resistance variants** identified across **6 genes** and **5 drug classes**
- Double **gyrA** mutations (D87N + A90V) predict high-level fluoroquinolone resistance
- **rpoB** S531L is the most common rifampicin resistance mutation globally
- **marR** G103S triggers a broad MDR response via AcrAB-TolC efflux upregulation
- **ompC** truncation (K163*) reduces outer-membrane permeability across multiple drug classes
- All variants at near-100% VAF, confirming a **clonal, single-isolate** genome

In [ ]:
# Final summary table
resist_df = filt_df[filt_df['FUNCTIONAL_CLASS'].isin(['NONSYNONYMOUS','NONSENSE'])].copy()

RESIST_DB_MINI = {
    ('gyrA', 'D87N'): ('Fluoroquinolone', 32),
    ('gyrA', 'A90V'): ('Fluoroquinolone', 16),
    ('rpoB', 'S531L'): ('Rifampicin', 256),
    ('marR', 'G103S'): ('Multidrug (MarA)', 8),
    ('acrA', 'I355V'): ('Multidrug (efflux)', 4),
    ('tolC', 'L457P'): ('Multidrug (efflux)', 6),
    ('ompC', 'K163*'): ('Broad spectrum', 4),
}

rows = []
for _, row in resist_df.iterrows():
    aa = row['AA_CHANGE']
    code = aa[aa.index('(')+1:aa.index(')')] if '(' in aa else aa
    entry = RESIST_DB_MINI.get((row['GENE'], code), ('—', '—'))
    rows.append({
        'Gene':        row['GENE'],
        'Mutation':    code,
        'Drug Class':  entry[0],
        'MIC Fold-↑': entry[1],
        'VAF (%)':     f"{row['AF']*100:.1f}%",
        'QUAL':        int(row['QUAL']),
    })

summary_df = pd.DataFrame(rows).sort_values('QUAL', ascending=False).reset_index(drop=True)
print('=== RESISTANCE VARIANT SUMMARY ===')
print(summary_df.to_string(index=False))